In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "brauer2006apes")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Braeuer_2006_inequityaverse_braeuer-fairness-all-2004.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)

df['study_id']="brauer2006apes"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns


In [3]:
##correct dyad name errors
df.at[70, 'competitor'] = 'frodo'
df.at[701, 'competitor'] = 'pini'
df.at[788, 'competitor'] = 'dokana'

In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

df['subject'] = df['subject'].str.rstrip()
df['competitor'] = df['competitor'].str.rstrip()

for x,y in zip(df_name['wrong'],df_name['right']):
    df['subject'].replace(x, y, inplace=True)
    df['competitor'].replace(x, y, inplace=True)

df['dyad']=df.subject.str.cat(df.competitor, sep='_')

In [5]:
df = df.rename(columns={"subject": "ape"})
df['role']='focal_participant'

df['role_2']='competitor'
df = df.rename(columns={"competitor": "ape_2"})

In [6]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

In [7]:
# df.columns
df.rename(columns={"ape": "participant", "ape_2":"participant_2",
                    "lat":"latency",
                    "scrat":"scratch",
                    "cond":"condition",
                    "ignor":"ignore",
                    'lat_per':'latency_percentage',
                    "scrat_yes":"scratch_yes"}, inplace=True)

In [8]:

brauer2006apes_standardized=df[['study_id', 'participant',
       'sex', 'role', 'participant_2','sex_2','role_2', 'species', 'dyad', 'condition', 
       'point', 'latency', 'away', 'scratch',
       'rock_lip_bang', 'eat', 'touch', 'ignore', 'time', 'latency_percentage',
       'scratch_yes', 'point_yes' ]]


In [9]:
comp_out_path_stand = os.path.join(out_pathway, 'brauer2006apes_standardized.csv')
brauer2006apes_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


In [10]:

names =brauer2006apes_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
brauer2006apes_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'brauer2006apes_glossary.csv')
brauer2006apes_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
